# Ridge Regression (L2)

Linear regression with **L2 (ridge) regularization**. Features are standardized inside a pipeline (scaler fit on training data only, then applied to test — no leakage). `RIDGE_ALPHA` controls shrinkage.

Same pipeline as the NN1 notebook — one `.parquet` read once and sliced by year, the trailing/cumulative window toggle, and a 4-column output (`permno, eom, target_w, prediction`) — but the model is swapped.

**Validation handling:** unlike the NN (which uses the validation block for early stopping), this model trains on the **combined train+valid years** (the full block NN1 reads) and predicts the test year. Same data span, no leakage — the test year is always strictly after everything used to fit.

**Coefficient statistics are saved** to `RIDGE_coef_stats.parquet`, same schema as `OLS_coef_stats.parquet`, so it drops into the accuracy/importance notebook with `MODEL="RIDGE"`. Computed once at fit time from the training design; nothing is refit downstream.

> **Read the ridge SE/t/p with care.** Ridge is a *biased (shrunk)* estimator. The SEs use the conditional-on-alpha ridge covariance `sigma^2 A^-1 (Z'Z) A^-1` with `A = Z'Z + alpha*I`, and `sigma^2` uses the effective degrees of freedom `df(alpha) = trace(Z(Z'Z+alpha I)^-1 Z') = p - alpha*trace(A^-1)`. These are Wald-type statistics: they quantify sampling variability at fixed alpha but do **not** account for shrinkage bias, so the p-values are **not** classical significance tests. Treat ridge t/p as a stability-and-scale signal; for "is this coefficient truly zero" use the OLS coefficient table. `alpha` and the effective df are stored alongside each row.

In [ ]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from numpy.linalg import pinv
from scipy.stats import t as t_dist

In [ ]:
# ---- configuration ---------------------------------------------------------
PARQUET_PATH = "US_GFD_FEATURES.parquet"
OUTPUT_PATH  = "RIDGE_predictions.parquet"
MODEL_DIR    = "RIDGE_models"   # fitted models saved here for feature importance
COEF_STATS_PATH = "RIDGE_coef_stats.parquet"   # per-model coef / SE / t / p / CI table
CONF            = 0.95                         # confidence level for the stored CIs

FIRST_YEAR = 1984
LAST_YEAR  = 2025
N_TRAIN    = 5
N_VALID    = 5
N_TEST     = 1

# "trailing" -> fixed N_TRAIN+N_VALID-year train window sliding forward
# "cumulative" -> expanding train from FIRST_YEAR up to the test year
SCHEME = "trailing"    # or "cumulative"

# ---- Ridge (L2) params ----
RIDGE_ALPHA = 10.0     # L2 penalty strength (higher = more shrinkage)

ID_COLS = ["permno","eom","gvkey","iid","cusip","tic","tpci","exchg",
           "shrcd","exchcd","sic","naics","trade_eom","acc_eom",
           "acc_datadate","rdq","date_buy_t1","date_sell_t1","ret_1m",
           "target","prc","prc_buy_t1","prc_sell_t1","n_days",
           "acc_age_m","target_w"]
TARGET = "target_w"

In [ ]:
# ---- resolve feature list from schema (no data read) -----------------------
schema = pq.read_schema(PARQUET_PATH)
FEATURES = [c for c in schema.names if c not in ID_COLS]
READ_COLS = ["permno", "gvkey", "eom", TARGET] + FEATURES
print(f"{len(FEATURES)} feature columns")

In [ ]:
# ---- read once, slice by year ---------------------------------------------
import time
_t = time.time()
FULL = pq.read_table(PARQUET_PATH, columns=READ_COLS).to_pandas()
FULL["eom"] = pd.to_datetime(FULL["eom"])
FULL = FULL.astype({**{c: "float32" for c in FEATURES}, TARGET: "float32"})
FULL["_year"] = FULL["eom"].dt.year
print(f"read {len(FULL):,} rows x {len(FULL.columns)} cols in {time.time()-_t:.1f}s, "
      f"{FULL.memory_usage(deep=True).sum()/1e9:.2f} GB")

_na = FULL[FEATURES].isna().to_numpy().sum()
_inf = np.isinf(FULL[FEATURES].to_numpy()).sum()
print(f"feature NaNs: {_na:,} | Infs: {_inf:,} | target NaNs: {FULL[TARGET].isna().sum():,}")
if _na or _inf:
    FULL[FEATURES] = FULL[FEATURES].replace([np.inf, -np.inf], np.nan)
    FULL[FEATURES] = FULL[FEATURES].fillna(0.5).astype("float32")

def slice_years(y0, y1):
    sub = FULL[FULL["_year"].between(y0, y1)]
    return sub[sub[TARGET].notna()]

In [ ]:
# ---- window schedule (trailing vs cumulative) ------------------------------
def make_windows(first_year, last_year, n_tr, n_va, n_te, scheme=SCHEME):
    wins = []
    test_start = first_year + n_tr + n_va
    while test_start <= last_year:
        va = (test_start - n_va, test_start - 1)
        te = (test_start, min(test_start + n_te - 1, last_year))
        if scheme == "trailing":
            tr = (test_start - n_tr - n_va, test_start - n_va - 1)
        elif scheme == "cumulative":
            tr = (first_year, test_start - n_va - 1)
        else:
            raise ValueError(f"unknown SCHEME {scheme!r}")
        wins.append((tr, va, te))
        test_start += n_te
    return wins

WINDOWS = make_windows(FIRST_YEAR, LAST_YEAR, N_TRAIN, N_VALID, N_TEST, SCHEME)
print(f"scheme={SCHEME!r}: {len(WINDOWS)} windows | first {WINDOWS[0]} | last {WINDOWS[-1]}")

In [ ]:
# ---- fit Ridge (standardize -> L2 linear) on the full train+valid block ----
def ridge_coef_stats(pipe, X, y, feature_names, test_year, alpha, conf=CONF):
    """Coefficient table for the Ridge fit, computed on the TRAINING design.
    SE uses the conditional-on-alpha ridge covariance  sigma^2 A^-1 (Z'Z) A^-1,
    A = Z'Z + alpha*I, with sigma^2 from the effective df df(alpha)=p-alpha*tr(A^-1).
    Nothing is refit. CAVEAT: ridge is biased, so these are Wald-type SEs/t/p that
    ignore shrinkage bias -- not classical significance tests."""
    sc  = pipe.named_steps["standardscaler"]
    rid = pipe.named_steps["ridge"]
    Z   = np.asarray(sc.transform(X), np.float64)
    Zc  = Z - Z.mean(0)
    yv  = np.asarray(y, np.float64)
    coef = np.ravel(rid.coef_).astype(np.float64)          # ridge (standardized) coefficients
    e    = yv - pipe.predict(X)                            # residuals on the fit data
    n, p = Zc.shape
    ZtZ  = Zc.T @ Zc
    A    = ZtZ + alpha * np.eye(p)
    Ainv = pinv(A)
    eff_df = float(p - alpha * np.trace(Ainv))             # trace(Z(Z'Z+aI)^-1 Z')
    dof  = max(n - eff_df - 1.0, 1.0)
    sig2 = float(e @ e) / dof
    cov  = sig2 * (Ainv @ ZtZ @ Ainv)                      # conditional-on-alpha covariance
    se   = np.sqrt(np.clip(np.diag(cov), 0.0, None))
    a0   = float(rid.intercept_)
    se0  = float(np.sqrt(sig2 / n))                        # intercept ~ mean(y)
    beta = np.concatenate([[a0], coef]); se_all = np.concatenate([[se0], se])
    tval = np.divide(beta, se_all, out=np.full_like(beta, np.nan), where=se_all > 0)
    pval = 2.0 * t_dist.sf(np.abs(tval), dof)
    tcr  = t_dist.ppf(1 - (1 - conf) / 2, dof)
    recon = float(np.max(np.abs((a0 + Z @ coef) - pipe.predict(X))))   # sanity: reproduces preds
    df = pd.DataFrame({
        "model_year": np.int32(test_year),
        "feature":    ["intercept"] + list(feature_names),
        "coef": beta, "abs_coef": np.abs(beta), "SE": se_all,
        "t_stat": tval, "p_value": pval,
        "ci_low": beta - tcr * se_all, "ci_high": beta + tcr * se_all,
        "n_obs": np.int64(n), "dof": np.float64(dof),
        "alpha": np.float64(alpha), "eff_df": np.float64(eff_df),
    })
    return df, recon


def fit_predict_window(Xtr, ytr, Xte, Xtr_only, ytr_only, Xva, yva,
                       test_year=None, model_dir=None, feature_names=None):
    # standardize features (fit on train only), then L2-penalized linear reg.
    import joblib
    m = make_pipeline(StandardScaler(), Ridge(alpha=RIDGE_ALPHA))
    m.fit(Xtr, ytr)
    if model_dir is not None:
        # the pipeline holds the scaler + fitted coefficients; the Ridge step's
        # .coef_ IS the (standardized) linear importance you can read directly.
        joblib.dump(m, f"{model_dir}/ridge_test{test_year}.joblib")
    # coefficient stats on the training design (cheap; Xtr already in memory)
    stats, recon = ridge_coef_stats(m, Xtr, ytr, feature_names, test_year, RIDGE_ALPHA)
    return m.predict(Xte), stats, recon


In [ ]:
# ---- roll through windows, write predictions + coefficient stats -----------
import json
from pathlib import Path
Path(MODEL_DIR).mkdir(exist_ok=True)
json.dump(FEATURES, open(f"{MODEL_DIR}/feature_names.json", "w"))
writer = None
coef_stats = []
recon_max = 0.0
for (tr, va, te) in WINDOWS:
    train_df = slice_years(tr[0], va[1])     # train+valid block = training data
    test_df  = slice_years(te[0], te[1])     # the held-out test year

    Xtr = train_df[FEATURES].to_numpy(np.float32)
    ytr = train_df[TARGET].to_numpy(np.float32)
    Xte = test_df[FEATURES].to_numpy(np.float32)

    # for models that use early stopping (xgboost), split off the valid years;
    # others ignore va and just use the whole train block.
    va_mask = train_df["eom"].dt.year.between(*va)
    tr_mask = train_df["eom"].dt.year.between(*tr)
    Xtr_only = train_df.loc[tr_mask, FEATURES].to_numpy(np.float32)
    ytr_only = train_df.loc[tr_mask, TARGET].to_numpy(np.float32)
    Xva = train_df.loc[va_mask, FEATURES].to_numpy(np.float32)
    yva = train_df.loc[va_mask, TARGET].to_numpy(np.float32)

    yhat, stats, recon = fit_predict_window(Xtr, ytr, Xte, Xtr_only, ytr_only, Xva, yva,
                                            te[0], MODEL_DIR, FEATURES)
    coef_stats.append(stats)
    recon_max = max(recon_max, recon)

    out = test_df[["permno", "gvkey", "eom", TARGET]].copy()
    out["prediction"] = yhat.astype(np.float32)
    tbl = pa.Table.from_pandas(out, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(OUTPUT_PATH, tbl.schema)
    writer.write_table(tbl)
    print(f"test {te[0]}: train {tr}, valid {va} | "
          f"n_train={len(Xtr):,} n_test={len(Xte):,} | pred mean={yhat.mean():.5f}")

if writer is not None:
    writer.close()
coef_df = pd.concat(coef_stats, ignore_index=True)
coef_df.to_parquet(COEF_STATS_PATH, index=False)
print(f"done -> {OUTPUT_PATH}")
print(f"coef stats: {coef_df.shape[0]:,} rows across {coef_df['model_year'].nunique()} "
      f"models (alpha={RIDGE_ALPHA}, eff_df~{coef_df['eff_df'].mean():.0f}) -> {COEF_STATS_PATH}")
print(f"max |hand-computed fitted - saved prediction| = {recon_max:.2e}")


In [ ]:
preds = pd.read_parquet(OUTPUT_PATH)
print(preds.shape, preds.columns.tolist())
preds.head()

In [ ]:
# ---- preview the stored coefficient statistics -----------------------------
cstats = pd.read_parquet(COEF_STATS_PATH)
print(cstats.shape, cstats.columns.tolist())
_last = cstats["model_year"].max()
(cstats[cstats["model_year"] == _last]
    .reindex(cstats[cstats["model_year"] == _last]["t_stat"].abs().sort_values(ascending=False).index)
    [["feature","coef","SE","t_stat","p_value","ci_low","ci_high","eff_df"]].head(12).round(4))
